In [ ]:
import $ivy.`org.apache.spark::spark-sql:4.1.4`

In [ ]:
import java.nio.file.{Files, Path, Paths}
import java.util.Comparator

val dirClases: Path = Paths.get("/tmp/clases_notebook")

// Empezar con la carpeta vacia en cada sesion del kernel
if (Files.exists(dirClases))
  Files.walk(dirClases).sorted(Comparator.reverseOrder[Path]()).forEach(p => Files.delete(p))
Files.createDirectories(dirClases)

// Copia a la carpeta todas las clases compiladas en el notebook hasta ahora
def exportarClases(): Unit =
  for {
    frame <- repl.sess.frames
    (nombre, bytes) <- frame.classloader.newFileDict
  } {
    val destino = dirClases.resolve(nombre.replace('.', '/') + ".class")
    Files.createDirectories(destino.getParent)
    Files.write(destino, bytes)
  }

// Exporta las clases y despues ejecuta la accion
def enCluster[T](accion: => T): T = {
  exportarClases()
  accion
}

In [ ]:
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("prueba_cluster_scala")
  .master("spark://spark-master:7077")
  .config("spark.driver.host", "jupyter")
  .config("spark.executor.cores", "1")
  .config("spark.executor.memory", "1g")
  .config("spark.cores.max", "3")
  .config("spark.repl.class.outputDir", dirClases.toString)
  .getOrCreate()

val sc = spark.sparkContext
println(s"Versión de Spark: ${spark.version}")

In [ ]:
import spark.implicits._
import org.apache.spark.sql.functions._

val pedidos = Seq(
  ("Madrid", "web", 120.5), ("Madrid", "tienda", 80.0),
  ("Bilbao", "web", 45.9), ("Bilbao", "web", 64.1),
  ("Sevilla", "tienda", 150.0)
).toDF("ciudad", "canal", "importe")

pedidos
  .groupBy("ciudad")
  .agg(count("*").as("pedidos"), round(sum("importe"), 2).as("total"))
  .orderBy("ciudad")
  .show()

In [ ]:
spark.read.parquet("/opt/datos/resumen_ventas").orderBy("tienda").show()

In [ ]:
import org.apache.spark.sql.functions.{col, udf}

enCluster {
  // 1. RDD con una funcion anonima
  val factor = 3
  println("RDD map: " + sc.parallelize(1 to 100, 6).map(_ * factor).sum())

  // 2. UDF definida en el notebook
  val conIva = udf((precio: Double) => math.round(precio * 1.21 * 100) / 100.0)
  Seq(10.0, 20.0).toDF("precio").withColumn("con_iva", conIva(col("precio"))).show()

  // 3. map sobre un Dataset
  println("Dataset map: " + spark.range(5).map(_ * 2).collect().mkString(","))
}

In [ ]:
spark.stop()